# Sbrodae Retrieval + Neural Rescoring + Neural FDR Benchmark

End-to-end evaluation of the trained dual-encoder on the *S. brodae* dataset, with thesis-grade dataset statistics and ANN diagnostics:

1. Build the **target + decoy search space** from `sbrodae_full_space search` (single CSV with an `is_decoy` flag).
2. Load **MS/MS spectra** from `20230608_..._Sbrodae.mgf` and the **ground-truth PSMs** from `sbrodae_PSM.csv`.
3. **Dataset statistics** for the thesis: per-side peptide-length distributions, PSM charge / m/z / length / q-value distributions, spectrum peak-count distribution, GT coverage in the DB.
4. **HNSW build + ANN diagnostics**: encode time, build time, index-on-disk size; latency / throughput sweep over `ef_search`; exact (brute-force) baseline for the recall ceiling.
5. **Stage-1 retrieval** -> Recall@k.
6. **Neural rescoring** with the **InstaNovo decoder** -> reranked Recall@k.
7. **Neural FDR**: top-1 precision FDR and target-decoy competition (reverse-inner decoys + KDE local FDR/PEP + 1% accepted set).

**Every intermediate result is exported to CSV** under `OUT_DIR`.

> Edit the two checkpoint paths in section *2* before running.

## 1. Environment, imports, paths

In [ ]:
from __future__ import annotations

import os, sys, json, time, gc
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import yaml
from torch.utils.data import DataLoader

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
for p in [REPO_ROOT, REPO_ROOT / "InstaNovo"]:
    if p.is_dir() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Repo root : {REPO_ROOT}")
print(f"Device    : {DEVICE}")

DATA_ROOT     = REPO_ROOT / "Data - Copy"
MGF_PATH      = REPO_ROOT / "20230608_EV_EVO_KK_FAIMS_2CV_Endurance_30SPD_1475_Sbrodae.mgf"
SEARCH_SPACE  = REPO_ROOT / "sbrodae_full_space search"
PSM_CSV       = DATA_ROOT / "sbrodae_PSM.csv"

# --- Checkpoints (placeholders -- edit after downloading from cloud) ---
SPEC_CKPT = REPO_ROOT / "checkpoints" / "model_spec.pt"   # <-- EDIT ME
PEP_CKPT  = REPO_ROOT / "checkpoints" / "model_pep.pt"    # <-- EDIT ME

OUT_DIR = DATA_ROOT / "benchmark_out" / "sbrodae"
OUT_DIR.mkdir(parents=True, exist_ok=True)
INDEX_DIR = OUT_DIR / "hnsw_index"
INDEX_DIR.mkdir(parents=True, exist_ok=True)

K_RETRIEVE        = 100
K_EVAL            = [1, 5, 10, 20, 50, 100]
RESCORE_TOP_N     = 50
STAGE1_TOP_K_FDR  = 50
FDR_CUTOFF        = 0.01
FIXED_MODS        = "C[UNIMOD:4]"
PSM_QVALUE_CUTOFF = 0.01
EF_SEARCH_SWEEP   = [16, 32, 64, 128, 256, 512]
RUN_EXACT_BASELINE = True   # set False if RAM/CPU is tight (full Q x D dot product)

for path, label in [(MGF_PATH, "MGF"), (SEARCH_SPACE, "Search space"), (PSM_CSV, "PSM CSV")]:
    print(f"  {label:<15s}: {path}  ({'OK' if path.exists() else 'MISSING'})")
print(f"  Out dir        : {OUT_DIR}")

## 2. Load InstaNovo backbone + dual encoders

* `instanovo_model` -- frozen pretrained backbone, used both as the spectrum encoder upstream and as the neural rescorer.
* `model_spec` -- `InstaSearchSpectrumEncoder` (projection head on top of the frozen backbone).
* `model_pep`  -- small transformer over the PTM-aware residue vocabulary.

> The notebook **warns** but proceeds if a checkpoint is missing -- numbers in that case are not meaningful.

In [ ]:
from instanovo.transformer.model import InstaNovo
from instanovo.utils.residues import ResidueSet
from instanovo.constants import LEGACY_PTM_TO_UNIMOD

from src.models.insta_search_spectrum_encoder import InstaSearchSpectrumEncoder
from src.models.peptide_encoder import PeptideEncoder
from src.utils.config import D_MODEL, N_HEADS, D_FF, N_LAYERS, EMBED_DIM, DROPOUT, MAX_PEPTIDE_LEN

INSTANOVO_CHECKPOINT = "instanovo-v1.2.0"

_orig_load = torch.load
torch.load = lambda *a, **kw: _orig_load(*a, **{**kw, "weights_only": False})
try:
    instanovo_model, instanovo_config = InstaNovo.from_pretrained(INSTANOVO_CHECKPOINT)
finally:
    torch.load = _orig_load
d_instanovo = int(instanovo_config["dim_model"])

residue_cfg_path = REPO_ROOT / "InstaNovo" / "instanovo" / "configs" / "residues" / "default.yaml"
with open(residue_cfg_path) as f:
    residue_cfg = yaml.safe_load(f)
residue_set = ResidueSet(
    residue_masses    = residue_cfg["residues"],
    residue_remapping = LEGACY_PTM_TO_UNIMOD,
)
NUM_AA = len(residue_set.vocab)
print(f"ResidueSet vocab size: {NUM_AA}")

model_spec = InstaSearchSpectrumEncoder(
    instanovo_model=instanovo_model, d_instanovo=d_instanovo,
    embed_dim=EMBED_DIM, freeze_encoder=True, dropout=DROPOUT,
).to(DEVICE)
model_pep = PeptideEncoder(
    d_model=D_MODEL, n_heads=N_HEADS, d_ff=D_FF, n_layers=N_LAYERS,
    embed_dim=EMBED_DIM, num_aa=NUM_AA,
).to(DEVICE)

if SPEC_CKPT.exists():
    state = torch.load(SPEC_CKPT, map_location=DEVICE)
    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]
    model_spec.load_state_dict(state, strict=False)
    print(f"Loaded spectrum encoder weights: {SPEC_CKPT}")
else:
    print(f"WARNING: spectrum-encoder checkpoint not found at {SPEC_CKPT}.")
    print("         Running with RANDOM projection-head weights -- results will be uncalibrated.")

if PEP_CKPT.exists():
    state = torch.load(PEP_CKPT, map_location=DEVICE)
    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]
    model_pep.load_state_dict(state, strict=False)
    print(f"Loaded peptide encoder weights : {PEP_CKPT}")
else:
    print(f"WARNING: peptide-encoder checkpoint not found at {PEP_CKPT}.")
    print("         Running with RANDOM peptide-encoder weights -- results will be uncalibrated.")

model_spec.eval(); model_pep.eval()

rescorer_model = instanovo_model.to(DEVICE).eval()
rescorer_model.residue_set.update_remapping(LEGACY_PTM_TO_UNIMOD)
print("InstaNovo rescorer ready.")

## 3. Build the search space (targets + decoys)

`sbrodae_full_space search` already contains both targets and decoys with an explicit `is_decoy` flag.

1. Drop peptides containing `U`, `O`, or `X` and length < 5 or > 40.
2. Apply fixed Carbamidomethyl-C (`C[UNIMOD:4]`).
3. Deduplicate within each side, then drop any decoys that collide with a target sequence.

In [ ]:
from src.retrieval.benchmarks import format_modified_sequence

MIN_LEN, MAX_LEN_PEP = 5, 40

raw = pd.read_csv(SEARCH_SPACE)
print(f"Search-space rows raw : {len(raw):,}")
print(f"Columns               : {list(raw.columns)}")

col_seq   = next(c for c in raw.columns if "sequence" in c.lower() or "trypsin" in c.lower())
col_prot  = next(c for c in raw.columns if "protein" in c.lower() or "origin"  in c.lower())
col_decoy = next(c for c in raw.columns if "decoy"   in c.lower())
raw = raw.rename(columns={col_seq: "peptide", col_prot: "protein", col_decoy: "is_decoy"})
raw["is_decoy"] = raw["is_decoy"].astype(str).str.lower().isin({"true", "1", "t", "yes"})
raw["peptide"]  = raw["peptide"].astype(str).str.strip()

n_raw_targets = int((~raw["is_decoy"]).sum())
n_raw_decoys  = int(raw["is_decoy"].sum())

def _clean(seqs):
    seen, out = set(), []
    for s in seqs:
        if not s or any(c in s for c in "UOX"):
            continue
        if not (MIN_LEN <= len(s) <= MAX_LEN_PEP):
            continue
        mod = format_modified_sequence(s, "", fixed_mods=FIXED_MODS)
        if mod and mod not in seen:
            seen.add(mod); out.append(mod)
    return out

target_seqs = _clean(raw.loc[~raw["is_decoy"], "peptide"])
decoy_seqs  = _clean(raw.loc[ raw["is_decoy"], "peptide"])
target_set  = set(target_seqs)
decoy_seqs  = [d for d in decoy_seqs if d not in target_set]
print(f"Raw targets / decoys     : {n_raw_targets:,} / {n_raw_decoys:,}")
print(f"Cleaned targets / decoys : {len(target_seqs):,} / {len(decoy_seqs):,}")

pd.DataFrame({"sequence": target_seqs}).to_csv(OUT_DIR / "db_targets.csv", index=False)
pd.DataFrame({"sequence": decoy_seqs }).to_csv(OUT_DIR / "db_decoys.csv",  index=False)
print("Wrote db_targets.csv and db_decoys.csv.")

## 4. Load PSMs + spectra and join

* PSMs filtered to q-value <= 1 % (confident GT).
* Spectra from MGF.
* `join_psm_with_spectra` aligns the surviving PSMs to their MS/MS spectra row-for-row.

In [ ]:
from src.retrieval.benchmarks import (
    load_psm_table,
    load_ms_spectra,
    join_psm_with_spectra,
)

psm_df_raw = pd.read_csv(PSM_CSV, low_memory=False)
n_psm_raw = len(psm_df_raw)
print(f"Raw PSM rows in CSV : {n_psm_raw:,}")

psm_df = load_psm_table(
    PSM_CSV,
    qvalue_cutoff = PSM_QVALUE_CUTOFF,
    fixed_mods    = FIXED_MODS,
)
psm_df.to_csv(OUT_DIR / "psm_table_filtered.csv", index=False)
print(f"Wrote psm_table_filtered.csv -- {len(psm_df):,} confident PSMs")

# How many GT modified sequences ALREADY live in the target DB (recall-ceiling diagnostic)?
gt_seqs        = sorted(set(psm_df["modified_sequence"].tolist()))
gt_in_db_pre   = sum(1 for s in gt_seqs if s in target_set)
gt_missing_pre = len(gt_seqs) - gt_in_db_pre
print(f"  GT unique sequences : {len(gt_seqs):,}")
print(f"    in target DB pre  : {gt_in_db_pre:,}  ({gt_in_db_pre/len(gt_seqs)*100:.1f}%)")
print(f"    NOT in DB pre     : {gt_missing_pre:,}  (will be augmented in)")

extra = [s for s in gt_seqs if s and s not in target_set]
if extra:
    target_seqs = target_seqs + extra
    target_set  = set(target_seqs)
    decoy_seqs  = [d for d in decoy_seqs if d not in target_set]

spectra_index = load_ms_spectra([MGF_PATH])
dataset, kept_df = join_psm_with_spectra(psm_df, spectra_index, residue_set)
kept_df.to_csv(OUT_DIR / "psm_table_joined.csv", index=False)
print(f"Wrote psm_table_joined.csv  -- {len(kept_df):,} PSMs joined to MGF spectra")

modified_sequences = kept_df["modified_sequence"].tolist()

## 4.5 Dataset statistics (thesis)

Compute and export descriptive statistics for the thesis. Distributions are saved as long-format CSVs (each row is a bucket) and a single-row `dataset_summary_stats.csv` collects the headline numbers.

**What's reported**

* **Peptide DB** -- per-side length distribution, fraction with C / M, total/unique counts.
* **PSMs** -- raw vs q-filtered vs joined counts, charge histogram, peptide-length histogram, m/z and q-value summaries, GT coverage in the DB.
* **Spectra** -- count per MS file, charge histogram, peak-count distribution, precursor m/z range.

In [ ]:
# -- helpers ------------------------------------------------------------
import re
_UNIMOD_RE = re.compile(r"\[UNIMOD:\d+\]")
def _strip_mods(seq: str) -> str:
    return _UNIMOD_RE.sub("", str(seq))

def _stats(arr):
    arr = np.asarray(arr)
    arr = arr[np.isfinite(arr)] if np.issubdtype(arr.dtype, np.number) else arr
    if arr.size == 0:
        return dict(n=0, mean=np.nan, std=np.nan, min=np.nan, q25=np.nan, median=np.nan, q75=np.nan, q99=np.nan, max=np.nan)
    return dict(
        n      = int(arr.size),
        mean   = float(np.mean(arr)),
        std    = float(np.std(arr)),
        min    = float(np.min(arr)),
        q25    = float(np.quantile(arr, 0.25)),
        median = float(np.median(arr)),
        q75    = float(np.quantile(arr, 0.75)),
        q99    = float(np.quantile(arr, 0.99)),
        max    = float(np.max(arr)),
    )

# -- DB: peptide-length distribution ------------------------------------
tgt_lens = np.array([len(_strip_mods(s)) for s in target_seqs])
dec_lens = np.array([len(_strip_mods(s)) for s in decoy_seqs ])
tgt_has_C = float(np.mean(["C" in _strip_mods(s) for s in target_seqs])) if target_seqs else 0.0
tgt_has_M = float(np.mean(["M" in _strip_mods(s) for s in target_seqs])) if target_seqs else 0.0

lens = sorted(set(tgt_lens.tolist()) | set(dec_lens.tolist()))
len_hist = pd.DataFrame({
    "length":     lens,
    "n_targets":  [int((tgt_lens == L).sum()) for L in lens],
    "n_decoys":   [int((dec_lens == L).sum()) for L in lens],
})
len_hist.to_csv(OUT_DIR / "stats_db_length_hist.csv", index=False)
print(f"Wrote stats_db_length_hist.csv  ({len(len_hist)} length buckets)")

# -- PSMs: charge / peplen histograms + summary ------------------------
psm_lens   = kept_df["modified_sequence"].map(lambda s: len(_strip_mods(s))).to_numpy()
psm_charge = kept_df["charge"].astype(int).to_numpy()
psm_mz     = kept_df["mz"].astype(float).to_numpy()
psm_q      = pd.to_numeric(kept_df["qvalue"], errors="coerce").to_numpy()

charge_hist = pd.Series(psm_charge).value_counts().sort_index().rename_axis("charge").reset_index(name="n_psms")
charge_hist.to_csv(OUT_DIR / "stats_psms_charge_hist.csv", index=False)

psmlen_hist = pd.Series(psm_lens).value_counts().sort_index().rename_axis("peptide_length").reset_index(name="n_psms")
psmlen_hist.to_csv(OUT_DIR / "stats_psms_peplen_hist.csv", index=False)
print(f"Wrote stats_psms_charge_hist.csv ({len(charge_hist)} rows)")
print(f"Wrote stats_psms_peplen_hist.csv ({len(psmlen_hist)} rows)")

# -- Spectra: peak counts, charges, precursor m/z ----------------------
peak_counts = np.array([len(v["mz_array"]) for v in spectra_index.values()])
spec_charge = np.array([int(v["charge"]) for v in spectra_index.values()])
spec_mz     = np.array([float(v["precursor_mz"]) for v in spectra_index.values()])

speccharge_hist = pd.Series(spec_charge).value_counts().sort_index().rename_axis("charge").reset_index(name="n_spectra")
speccharge_hist.to_csv(OUT_DIR / "stats_spectra_charge_hist.csv", index=False)

peak_bins = np.linspace(0, peak_counts.max() if peak_counts.size else 1, 41)
peak_hist, _ = np.histogram(peak_counts, bins=peak_bins)
pd.DataFrame({
    "peak_bin_left":  peak_bins[:-1],
    "peak_bin_right": peak_bins[1:],
    "n_spectra":      peak_hist.astype(int),
}).to_csv(OUT_DIR / "stats_spectra_peak_hist.csv", index=False)
print("Wrote stats_spectra_charge_hist.csv, stats_spectra_peak_hist.csv")

# -- One-row summary across all three (thesis-grade) -------------------
len_t = _stats(tgt_lens); len_d = _stats(dec_lens)
len_p = _stats(psm_lens); peaks = _stats(peak_counts); mz_p = _stats(psm_mz); mz_s = _stats(spec_mz)

gt_in_db_post = sum(1 for s in gt_seqs if s in target_set)

ds_summary = {
    "n_raw_target_rows":           n_raw_targets,
    "n_raw_decoy_rows":            n_raw_decoys,
    "n_unique_targets_cleaned":    len(target_seqs),
    "n_unique_decoys_cleaned":     len(decoy_seqs),
    "target_len_mean":             len_t["mean"],
    "target_len_std":              len_t["std"],
    "target_len_min":              len_t["min"],
    "target_len_median":           len_t["median"],
    "target_len_max":              len_t["max"],
    "decoy_len_mean":              len_d["mean"],
    "decoy_len_median":            len_d["median"],
    "frac_target_with_C":          tgt_has_C,
    "frac_target_with_M":          tgt_has_M,
    "n_psms_raw":                  n_psm_raw,
    "n_psms_after_qfilter":        len(psm_df),
    "n_psms_joined_to_mgf":        len(kept_df),
    "frac_psms_joined":            len(kept_df) / max(len(psm_df), 1),
    "n_unique_gt_sequences":       len(gt_seqs),
    "gt_in_db_before_augment":     gt_in_db_pre,
    "gt_in_db_after_augment":      gt_in_db_post,
    "gt_coverage_after_aug":       gt_in_db_post / max(len(gt_seqs), 1),
    "psm_charge_mean":             float(np.mean(psm_charge)) if psm_charge.size else np.nan,
    "psm_charge_min":              int(psm_charge.min()) if psm_charge.size else 0,
    "psm_charge_max":              int(psm_charge.max()) if psm_charge.size else 0,
    "psm_len_mean":                len_p["mean"],
    "psm_len_median":              len_p["median"],
    "psm_len_max":                 len_p["max"],
    "psm_mz_mean":                 mz_p["mean"],
    "psm_mz_min":                  mz_p["min"],
    "psm_mz_max":                  mz_p["max"],
    "psm_qvalue_max_used":         float(np.nanmax(psm_q)) if psm_q.size else np.nan,
    "n_spectra_indexed_from_mgf":  len(spectra_index),
    "spectra_peak_count_mean":     peaks["mean"],
    "spectra_peak_count_median":   peaks["median"],
    "spectra_peak_count_q99":      peaks["q99"],
    "spectra_peak_count_max":      peaks["max"],
    "spectra_precursor_mz_min":    mz_s["min"],
    "spectra_precursor_mz_mean":   mz_s["mean"],
    "spectra_precursor_mz_max":    mz_s["max"],
}
pd.DataFrame([ds_summary]).to_csv(OUT_DIR / "dataset_summary_stats.csv", index=False)
print("\nWrote dataset_summary_stats.csv")
for k, v in ds_summary.items():
    print(f"  {k:<32s} = {v}")

## 5. Build the HNSW index (encode + index, separately timed)

We split `build_database_index` into its two phases so we can report them separately for the thesis:

1. **Encode** every DB peptide with the peptide encoder -> `db_emb` (also used in section 6.5 for the exact-baseline comparison).
2. **Build** the FAISS `IndexHNSWFlat` over `db_emb`.
3. **Save** the index and measure its on-disk size.

In [ ]:
from src.retrieval import HNSWConfig, HNSWIndex
from src.data.preprocess import preprocess_peptide_residueset
from tqdm.auto import tqdm

hnsw_cfg = HNSWConfig(
    embed_dim       = EMBED_DIM,
    M               = 32,
    ef_construction = 200,
    ef_search       = 128,
    k_retrieve      = K_RETRIEVE,
    index_dir       = str(INDEX_DIR),
    index_name      = "sbrodae_hnsw",
)

# 1. Encode the full target+decoy DB.
db_seqs     = list(target_seqs) + list(decoy_seqs)
is_decoy_db = np.concatenate([
    np.zeros(len(target_seqs), dtype=bool),
    np.ones(len(decoy_seqs),  dtype=bool),
])
print(f"Encoding {len(db_seqs):,} peptides ({len(target_seqs):,} targets + {len(decoy_seqs):,} decoys) ...")

ENCODE_BATCH = 1024
embeddings = []
t_encode_start = time.time()
with torch.no_grad():
    for start in tqdm(range(0, len(db_seqs), ENCODE_BATCH), desc="Encoding DB"):
        chunk = db_seqs[start:start + ENCODE_BATCH]
        toks = np.stack([
            preprocess_peptide_residueset(s, residue_set, max_len=MAX_PEPTIDE_LEN)
            for s in chunk
        ])
        z = model_pep(torch.as_tensor(toks, dtype=torch.int64, device=DEVICE))
        embeddings.append(z.cpu().float().numpy())
encode_time_s = time.time() - t_encode_start
db_emb = np.vstack(embeddings).astype(np.float32, copy=False)
print(f"Encode time      : {encode_time_s:.2f}s  ({len(db_seqs)/encode_time_s:,.0f} peptides/s)")
print(f"db_emb shape     : {db_emb.shape}  dtype={db_emb.dtype}")

# 2. Build the HNSW index.
t_build_start = time.time()
index = HNSWIndex(hnsw_cfg).build(db_emb)
hnsw_build_time_s = time.time() - t_build_start
print(f"HNSW build time  : {hnsw_build_time_s:.2f}s  ({len(db_seqs)/hnsw_build_time_s:,.0f} peptides/s)")

# 3. Persist + size.
index.save()
index_disk_bytes = sum(p.stat().st_size for p in INDEX_DIR.glob(f"{hnsw_cfg.index_name}*"))
print(f"Index on disk    : {index_disk_bytes/1e6:.1f} MB")
index.info()

pd.DataFrame({
    "db_row":   np.arange(len(db_seqs)),
    "sequence": db_seqs,
    "is_decoy": is_decoy_db,
}).to_csv(OUT_DIR / "db_index_contents.csv", index=False)

ann_build_stats = {
    "n_db_vectors":          int(db_emb.shape[0]),
    "embed_dim":             int(db_emb.shape[1]),
    "hnsw_M":                hnsw_cfg.M,
    "hnsw_ef_construction":  hnsw_cfg.ef_construction,
    "hnsw_ef_search_default":hnsw_cfg.ef_search,
    "hnsw_max_layer":        int(index.index.hnsw.max_level),
    "encode_time_s":         round(encode_time_s, 4),
    "encode_throughput_per_s": round(len(db_seqs) / max(encode_time_s, 1e-9), 2),
    "hnsw_build_time_s":     round(hnsw_build_time_s, 4),
    "hnsw_build_throughput_per_s": round(len(db_seqs) / max(hnsw_build_time_s, 1e-9), 2),
    "index_disk_size_mb":    round(index_disk_bytes / 1e6, 3),
    "index_dir":             str(INDEX_DIR),
}
pd.DataFrame([ann_build_stats]).to_csv(OUT_DIR / "ann_build_stats.csv", index=False)
print("Wrote ann_build_stats.csv and db_index_contents.csv.")

## 6. Stage-1 retrieval (HNSW cosine top-k)

In [ ]:
from src.retrieval.search import extract_embeddings, retrieve_batch

spec_emb = extract_embeddings(
    model_spec, dataset, mode="spectrum",
    batch_size=256, device=DEVICE, desc="Encoding query spectra",
)
spec_emb = spec_emb.astype(np.float32, copy=False)

t0 = time.time()
stage1_candidates, stage1_ranks = retrieve_batch(
    spec_emb,
    modified_sequences,
    index,
    db_seqs,
    k=K_RETRIEVE,
)
stage1_elapsed = time.time() - t0
Q = len(stage1_ranks)
print(f"Stage-1 retrieval: {Q:,} queries in {stage1_elapsed:.2f}s "
      f"({stage1_elapsed/Q*1000:.2f} ms/query, {Q/stage1_elapsed:,.0f} qps)")

recall_stage1 = {k: float((stage1_ranks <= k).mean()) for k in K_EVAL if k <= K_RETRIEVE}
pd.DataFrame({
    "k":             list(recall_stage1.keys()),
    "recall_stage1": list(recall_stage1.values()),
}).to_csv(OUT_DIR / "recall_at_k_stage1.csv", index=False)
print("\nRecall@k (Stage-1):")
for k, r in recall_stage1.items():
    print(f"  R@{k:<3d} = {r:.4f}")

db_seq_to_uid = {s: i for i, s in enumerate(db_seqs)}
rows = []
for qi, cands in enumerate(stage1_candidates):
    row = kept_df.iloc[qi]
    for rank, (seq, score) in enumerate(cands, start=1):
        uid = db_seq_to_uid.get(seq, -1)
        rows.append({
            "query_idx":     qi,
            "scan_id":       int(row["scan_id"]),
            "ms_file":       row["ms_file"],
            "true_sequence": modified_sequences[qi],
            "rank":          rank,
            "retrieved_seq": seq,
            "cosine":        float(score),
            "is_decoy":      bool(is_decoy_db[uid]) if uid >= 0 else False,
            "is_correct":    seq == modified_sequences[qi],
        })
pd.DataFrame(rows).to_csv(OUT_DIR / "stage1_topk_candidates.csv", index=False)
print(f"Wrote stage1_topk_candidates.csv -- {len(rows):,} rows")

## 6.5 ANN diagnostics -- latency, throughput, exact baseline (thesis)

Two diagnostics here:

1. **`ef_search` sweep** -- runtime knob that trades recall against latency without rebuilding the index. For each value we time `index.search` on all query spectra (best of 3 trials) and recompute Recall@k from the returned indices.
2. **Exact (brute-force) baseline** -- compute the full `(Q x D) cos sim` matrix once for a recall ceiling. Difference vs HNSW = ANN approximation loss.

Set `RUN_EXACT_BASELINE = False` in section 1 to skip the exact part on tight memory.

In [ ]:
# ---------- ef_search sweep --------------------------------------------
true_uids = np.array(
    [db_seq_to_uid.get(s, -1) for s in modified_sequences],
    dtype=np.int64,
)
n_with_gt_in_db = int((true_uids >= 0).sum())
print(f"Queries whose GT exists in DB: {n_with_gt_in_db:,} / {Q:,}")

def _recall_from_indices(idxs: np.ndarray, true_uids: np.ndarray, ks):
    out = {}
    matched = (idxs == true_uids[:, None])
    for k in ks:
        out[k] = float(matched[:, :k].any(axis=1).mean())
    return out

sweep_rows = []
for ef in EF_SEARCH_SWEEP:
    if ef < max(K_EVAL):
        print(f"  skipping ef={ef} (must be >= max k={max(K_EVAL)})")
        continue
    index.set_ef_search(ef)
    times = []
    dists, idxs = None, None
    for trial in range(3):
        t0 = time.time()
        dists, idxs = index.search(spec_emb, k=K_RETRIEVE)
        times.append(time.time() - t0)
    best_t = min(times)
    rec    = _recall_from_indices(idxs, true_uids, K_EVAL)
    sweep_rows.append({
        "ef_search":     ef,
        "total_time_s":  round(best_t, 4),
        "ms_per_query":  round(best_t / Q * 1000, 4),
        "queries_per_s": round(Q / best_t, 1),
        **{f"recall@{k}": round(v, 4) for k, v in rec.items()},
    })
    rec_str = "  ".join(f"R@{k}={v:.3f}" for k, v in rec.items())
    print(f"  ef={ef:<4d}  {best_t*1000/Q:6.2f} ms/q  {Q/best_t:7,.0f} qps   {rec_str}")

sweep_df = pd.DataFrame(sweep_rows)
sweep_df.to_csv(OUT_DIR / "ann_latency_sweep.csv", index=False)
print(f"\nWrote ann_latency_sweep.csv ({len(sweep_df)} rows)")

# Restore the default ef_search so downstream sections use the canonical setting.
index.set_ef_search(hnsw_cfg.ef_search)

# ---------- Exact baseline (FAISS IndexFlatIP) --------------------------
if RUN_EXACT_BASELINE:
    import faiss
    print("\nBuilding exact (brute-force) FlatIP index for recall ceiling...")
    t_exact_build = time.time()
    flat = faiss.IndexFlatIP(EMBED_DIM)
    flat.add(db_emb)
    exact_build_s = time.time() - t_exact_build
    t_exact_search = time.time()
    _, idx_exact = flat.search(spec_emb, K_RETRIEVE)
    exact_search_s = time.time() - t_exact_search
    rec_exact = _recall_from_indices(idx_exact, true_uids, K_EVAL)
    print(f"  exact build  : {exact_build_s:.2f}s")
    print(f"  exact search : {exact_search_s:.2f}s  ({exact_search_s/Q*1000:.2f} ms/q)")
    for k in K_EVAL:
        print(f"  R@{k:<3d}  exact={rec_exact[k]:.4f}   hnsw(default ef)={recall_stage1.get(k, float('nan')):.4f}")

    exact_rows = [{
        "k":                 k,
        "recall_exact":      round(rec_exact[k], 4),
        "recall_hnsw_default": round(recall_stage1.get(k, float("nan")), 4),
        "ann_loss_vs_exact": round(rec_exact[k] - recall_stage1.get(k, float("nan")), 4),
    } for k in K_EVAL if k <= K_RETRIEVE]
    exact_df = pd.DataFrame(exact_rows)
    exact_df.to_csv(OUT_DIR / "ann_exact_vs_hnsw.csv", index=False)
    print(f"Wrote ann_exact_vs_hnsw.csv ({len(exact_df)} rows)")

    extra_row = pd.DataFrame([{
        "variant":              "exact_flat_ip",
        "build_time_s":         round(exact_build_s, 4),
        "search_time_total_s":  round(exact_search_s, 4),
        "search_ms_per_query":  round(exact_search_s / Q * 1000, 4),
        "search_queries_per_s": round(Q / exact_search_s, 1),
    }])
    extra_row.to_csv(OUT_DIR / "ann_exact_timing.csv", index=False)
    print("Wrote ann_exact_timing.csv")
else:
    rec_exact     = None
    exact_build_s = None
    exact_search_s = None
    print("Exact baseline skipped (RUN_EXACT_BASELINE=False).")

## 7. Neural rescoring with InstaNovo decoder

Top-`RESCORE_TOP_N` Stage-1 candidates rescored by `instanovo_score` (mean log-prob under teacher forcing). The reranked block replaces the original head; the Stage-1 tail is appended so Recall@k for k > N is never degraded.

In [ ]:
from src.retrieval.rerank import rescore_and_rerank

pres3 = dataset.pres.float()
pres2 = torch.stack([pres3[:, 2], pres3[:, 1]], dim=1)

t_rescore_start = time.time()
reranked_candidates, rescored_ranks = rescore_and_rerank(
    model              = rescorer_model,
    stage1_candidates  = stage1_candidates,
    spec_tensor        = dataset.specs,
    pre_tensor         = pres2,
    ground_truth_seqs  = modified_sequences,
    rescore_top_n      = RESCORE_TOP_N,
    rescore_batch      = 8,
    reduction          = "mean",
)
rescore_elapsed = time.time() - t_rescore_start
print(f"Rescoring: {Q:,} queries x top-{RESCORE_TOP_N} in {rescore_elapsed:.1f}s "
      f"({rescore_elapsed/Q*1000:.1f} ms/q, {Q*RESCORE_TOP_N/rescore_elapsed:,.0f} cand/s)")

recall_rescored = {k: float((rescored_ranks <= k).mean()) for k in K_EVAL if k <= K_RETRIEVE}
pd.DataFrame({
    "k":               list(recall_rescored.keys()),
    "recall_stage1":   [recall_stage1[k]   for k in recall_rescored],
    "recall_rescored": [recall_rescored[k] for k in recall_rescored],
    "delta":           [recall_rescored[k] - recall_stage1[k] for k in recall_rescored],
}).to_csv(OUT_DIR / "recall_at_k.csv", index=False)
print("Recall@k (stage1 -> rescored):")
for k in recall_rescored:
    print(f"  R@{k:<3d}: {recall_stage1[k]:.4f} -> {recall_rescored[k]:.4f} (delta {recall_rescored[k] - recall_stage1[k]:+.4f})")

rows = []
for qi, cands in enumerate(reranked_candidates):
    row = kept_df.iloc[qi]
    for rank, (seq, score) in enumerate(cands, start=1):
        uid = db_seq_to_uid.get(seq, -1)
        rows.append({
            "query_idx":     qi,
            "scan_id":       int(row["scan_id"]),
            "ms_file":       row["ms_file"],
            "true_sequence": modified_sequences[qi],
            "rank":          rank,
            "rescored_seq":  seq,
            "neural_score":  float(score),
            "is_decoy":      bool(is_decoy_db[uid]) if uid >= 0 else False,
            "is_correct":    seq == modified_sequences[qi],
            "rescored_within_topN": rank <= RESCORE_TOP_N,
        })
pd.DataFrame(rows).to_csv(OUT_DIR / "rescored_topk_candidates.csv", index=False)
print(f"Wrote rescored_topk_candidates.csv -- {len(rows):,} rows")

## 8. Neural top-1 FDR (precision against unique-target DB)

In [ ]:
from src.retrieval.search import compute_fdr

loader = DataLoader(dataset, batch_size=256, num_workers=0)

fdr_top1 = compute_fdr(
    model_spec          = model_spec,
    model_pep           = model_pep,
    loader              = loader,
    device              = DEVICE,
    modified_sequences  = modified_sequences,
    rescorer_model      = rescorer_model,
    peptide_residue_set = residue_set,
    stage1_top_k        = STAGE1_TOP_K_FDR,
    score_mode          = "geometric_mean",
)

pd.DataFrame(fdr_top1["threshold_results"]).to_csv(OUT_DIR / "top1_fdr_thresholds.csv", index=False)
print(f"Wrote top1_fdr_thresholds.csv -- {len(fdr_top1['threshold_results']):,} rows")

n = fdr_top1["n_total"]
top1_df = pd.DataFrame({
    "query_idx":          np.arange(n),
    "scan_id":            kept_df["scan_id"].astype(int).values[:n],
    "ms_file":            kept_df["ms_file"].values[:n],
    "true_sequence":      fdr_top1["true_sequences"],
    "predicted_sequence": fdr_top1["top1_sequences"],
    "correct":            fdr_top1["correct_mask"],
})
for opt_key, out_key in [
    ("top1_scores",      "top1_score"),
    ("mean_logp",        "mean_logp"),
    ("true_pair_scores", "true_pair_score"),
]:
    arr = fdr_top1.get(opt_key)
    if arr is not None and len(arr) == n:
        top1_df[out_key] = np.asarray(arr)
top1_df.to_csv(OUT_DIR / "top1_predictions.csv", index=False)
print(f"Wrote top1_predictions.csv -- {len(top1_df):,} rows")

print("\nNeural top-1 summary:")
print(f"  precision_top1 = {fdr_top1['precision_top1']:.4f}")
print(f"  fdr_top1       = {fdr_top1['fdr_top1']:.4f}")
print(f"  n_correct      = {fdr_top1['n_correct_top1']:,} / {n:,}")

## 9. Neural Target-Decoy FDR (TDC + q-values + 1% accepted set)

In [ ]:
from src.retrieval.search import compute_tda_fdr

tda = compute_tda_fdr(
    model_spec          = model_spec,
    model_pep           = model_pep,
    loader              = loader,
    device              = DEVICE,
    modified_sequences  = modified_sequences,
    decoy_strategy      = "reverse-inner",
    rescorer_model      = rescorer_model,
    peptide_residue_set = residue_set,
    stage1_top_k        = STAGE1_TOP_K_FDR,
    score_mode          = "geometric_mean",
    fdr_cutoff          = FDR_CUTOFF,
)

comp_psms = tda.get("competition_psms")
if comp_psms is not None:
    comp_psms.to_csv(OUT_DIR / "competition_psms.csv", index=False)
    print(f"Wrote competition_psms.csv -- {len(comp_psms):,} rows")

acc_psms = tda.get("accepted_psms")
if acc_psms is not None:
    acc_psms.to_csv(OUT_DIR / f"accepted_psms_{int(FDR_CUTOFF*100)}pct.csv", index=False)
    print(f"Wrote accepted_psms_{int(FDR_CUTOFF*100)}pct.csv -- {len(acc_psms):,} rows")

n_tda = int(np.asarray(tda["final_scores"] if "final_scores" in tda else tda["top1_scores"]).shape[0])
tda_long = pd.DataFrame({
    "row":      np.arange(n_tda),
    "score":    np.asarray(tda.get("final_scores", tda.get("top1_scores"))),
    "is_decoy": np.asarray(tda.get("final_is_decoy", tda.get("top1_is_decoy"))).astype(bool),
    "sequence": list(tda.get("final_sequences", tda.get("top1_origin_sequence", [""]*n_tda))),
    "qvalue":   np.asarray(tda["qvalues"]),
})
if "pep" in tda:
    tda_long["pep"] = np.asarray(tda["pep"])
tda_long.to_csv(OUT_DIR / "tda_per_spectrum.csv", index=False)
print(f"Wrote tda_per_spectrum.csv -- {len(tda_long):,} rows")

if "threshold_results" in tda:
    pd.DataFrame(tda["threshold_results"]).to_csv(OUT_DIR / "tda_threshold_results.csv", index=False)
    print(f"Wrote tda_threshold_results.csv -- {len(tda['threshold_results']):,} rows")

if "sorted_scores" in tda and "sorted_qvalues" in tda:
    pd.DataFrame({
        "score":  np.asarray(tda["sorted_scores"]),
        "qvalue": np.asarray(tda["sorted_qvalues"]),
    }).to_csv(OUT_DIR / "tda_sorted_qvalue_curve.csv", index=False)
    print("Wrote tda_sorted_qvalue_curve.csv")

print("\nNeural TDA summary:")
print(f"  n_total          = {tda.get('n_total', n_tda)}")
print(f"  n_target_top1    = {tda.get('n_target_top1', '?')}")
print(f"  n_decoy_top1     = {tda.get('n_decoy_top1', '?')}")
if 'pi0' in tda:
    print(f"  pi0              = {tda['pi0']:.4f}")
print(f"  accepted @ {FDR_CUTOFF*100:.0f}%  = {tda.get('n_accepted_at_cutoff', '?')}")
if 'accepted_score_cutoff' in tda:
    print(f"  score cutoff     = {tda['accepted_score_cutoff']:.6f}")

## 10. Headline summary CSV (all metrics + ANN timing)

In [ ]:
summary = {
    "dataset":            "sbrodae",
    "mgf":                MGF_PATH.name,
    "search_space_file":  SEARCH_SPACE.name,
    "psm_csv":            PSM_CSV.name,
    # data sizes
    "n_psms_raw":         int(n_psm_raw),
    "n_psms_after_qfilter": int(len(psm_df)),
    "n_psms_joined":      int(len(kept_df)),
    "n_targets_in_db":    int((~is_decoy_db).sum()),
    "n_decoys_in_db":     int(is_decoy_db.sum()),
    "n_queries":          int(Q),
    "n_spectra_in_mgf":   int(len(spectra_index)),
    # config
    "k_retrieve":         K_RETRIEVE,
    "rescore_top_n":      RESCORE_TOP_N,
    "stage1_top_k_fdr":   STAGE1_TOP_K_FDR,
    "fdr_cutoff":         FDR_CUTOFF,
    # ANN timing
    "encode_db_time_s":   ann_build_stats["encode_time_s"],
    "hnsw_build_time_s":  ann_build_stats["hnsw_build_time_s"],
    "index_disk_size_mb": ann_build_stats["index_disk_size_mb"],
    "stage1_retrieval_s": round(stage1_elapsed, 3),
    "stage1_ms_per_q":    round(stage1_elapsed / Q * 1000, 3),
    "stage1_qps":         round(Q / stage1_elapsed, 1),
    "rescore_time_s":     round(rescore_elapsed, 3),
    # quality
    "precision_top1_neural":   float(fdr_top1["precision_top1"]),
    "fdr_top1_neural":         float(fdr_top1["fdr_top1"]),
    "n_correct_top1":          int(fdr_top1["n_correct_top1"]),
    "n_target_top1_tda":       int(tda.get("n_target_top1", 0)),
    "n_decoy_top1_tda":        int(tda.get("n_decoy_top1", 0)),
    "pi0_tda":                 float(tda.get("pi0", float("nan"))),
    "n_accepted_at_cutoff":    int(tda.get("n_accepted_at_cutoff", 0)),
    "accepted_score_cutoff":   float(tda.get("accepted_score_cutoff", float("nan"))),
}
for k, v in recall_stage1.items():
    summary[f"recall_stage1_at_{k}"]   = v
for k, v in recall_rescored.items():
    summary[f"recall_rescored_at_{k}"] = v
if RUN_EXACT_BASELINE and rec_exact is not None:
    for k, v in rec_exact.items():
        summary[f"recall_exact_at_{k}"] = v
    summary["exact_search_s"] = round(exact_search_s, 3)

summary_df = pd.DataFrame([summary])
summary_df.to_csv(OUT_DIR / "summary.csv", index=False)
print("Wrote summary.csv:")
for col in summary_df.columns:
    print(f"  {col:<28s} = {summary_df.iloc[0][col]}")

## 11. Index of all output CSVs

In [ ]:
out_files = sorted(OUT_DIR.glob("*.csv"))
manifest = pd.DataFrame({
    "file":     [p.name for p in out_files],
    "path":     [str(p) for p in out_files],
    "size_kb":  [round(p.stat().st_size / 1024, 1) for p in out_files],
})
manifest.to_csv(OUT_DIR / "_manifest.csv", index=False)
manifest